# Data Analysis & Modeling: Will a Player Return?

**Original interview mock, not a reported Roblox question or an official CodeSignal assessment.**
The format follows the interview description in your [shared conversation](https://chatgpt.com/share/6ad92cab-a9dc-83e8-b72e-654063228d54): **5 minutes introduction + 35 minutes coding + 5 minutes Q&A**. All data are synthetic; do not interpret these results as Roblox player behavior.

## The interviewer prompt

You work on a game platform. At the start of day 0, predict whether each player will return during days 0 through 6, using only information available before day 0.

**Problem 1 — Data analysis (14 minutes).** Load two CSVs, inspect their quality, clean the session log, and build one row per player. Report the percentage of players with no historical sessions and average historical play time per device, including inactive players.

**Problem 2 — Modeling (21 minutes).** Select valid features, create a holdout set, train a simple classification baseline, evaluate it against a no-skill baseline, and predict for two new players. Explain your choices as you code. No deep learning or from-scratch optimizer is required.

| Coding time | Task |
| --- | --- |
| 0–4 min | Load and inspect |
| 4–9 min | Clean the session log |
| 9–14 min | Aggregate and join |
| 14–18 min | Select features and split |
| 18–27 min | Preprocess and train |
| 27–32 min | Evaluate |
| 32–35 min | Predict and summarize |

These are suggested checkpoints, not official sub-round timings. Ask during the introduction: What is one example? When is a prediction made? Which mistake is more costly?


**Practice version.** Replace each NotImplementedError cell with your code, then run its supplied check cell. Imports, inputs and checks are provided. The intentional TODO stops mean Run All will not finish until you implement the steps. Use codesignal_roblox_retention_solution.ipynb only when reviewing.


## Data contract — supplied by the interviewer

**players.csv:** one row per player; attributes are snapshots taken before day 0. All players' outcome windows have completed, so labels are known. Every player belongs in the analysis, even if they have no session history.

| Column | Meaning |
| --- | --- |
| user_id | Unique identifier; not a numeric predictor |
| account_age_days | Account age at the cutoff; may be missing |
| country, device | Categorical attributes at the cutoff; may be missing |
| returned_next_week | Label: 1 if the player returns during days 0–6, otherwise 0 |
| next_week_minutes | Total minutes during days 0–6; present in the offline export, **unavailable at prediction time** |

**sessions.csv:** one row per logged session before cleaning.

| Column | Meaning |
| --- | --- |
| session_id | Session identifier; repeated exports may contain exact duplicate rows |
| user_id | Player identifier, always referencing a supplied player |
| day_offset | Relative to the cutoff: only **-7 through -1** are valid historical features |
| minutes_played | Session duration; blank or a negative sentinel means unknown, not zero |
| completed | 0/1 session-completion indicator; valid even if duration is unknown |

All session rows are available in this small batch exercise; the export deliberately contains some post-cutoff rows. For this mock, distinct players are assumed to be independent samples from the same population. A future-deployment evaluation would instead need multiple cutoffs and a chronological holdout.


## Provided setup

Open this notebook from the workspace root in VS Code or Jupyter. Select the Python environment at **.venv-codesignal/bin/python** (or another environment with pandas, NumPy and scikit-learn). The two CSVs live in **data/codesignal_roblox/**. No network or GPU is used during the exercise. For a fresh local setup, see **CODESIGNAL_PRACTICE.md**; the real CodeSignal environment has its own libraries.


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score, confusion_matrix,
)

SEED = 42
DATA_DIR = Path("data/codesignal_roblox")
assert (DATA_DIR / "players.csv").exists(), "Run from the workspace root, or change DATA_DIR to your CSV folder."
assert (DATA_DIR / "sessions.csv").exists()


## Problem 1 · Step 1 — Load and inspect (0–4 min)

Load **players** and **sessions**. Inspect shapes, sample rows, dtypes, missingness and numeric summaries. Check player uniqueness, duplicate sessions, invalid durations, out-of-window records, and label balance. Do not start dropping every incomplete row.

**Say before coding:** "I will first check the unit of observation and data quality. The label is at player level, but the event log has many rows per player."

**Possible interviewer questions**

1. What does one training example represent?
2. Which columns must not become features?
3. What do you check before filling missing values?


In [14]:
# Step 1: write your implementation here.
# Create the variables named in the task above.
players = pd.read_csv(DATA_DIR / 'players.csv')
sessions = pd.read_csv(DATA_DIR / 'sessions.csv')

display(players.head())
display(sessions.head())

print("duplicated", players.duplicated().sum())
print("duplicated", sessions.duplicated().sum())

print("out of window", (~sessions['day_offset'].between(-7,-1)).sum())

print("label distribution:",players["returned_next_week"].value_counts(normalize=True))

print("invalidate duration:",sessions['minutes_played'].lt(0).sum() )

#raise NotImplementedError("Complete Step 1, then run the checks below.")


,user_id,account_age_days,country,device,returned_next_week,next_week_minutes
0,user_0001,184.0,US,mobile,0,0.00
1,user_0002,6.0,US,mobile,0,0.00
2,user_0003,56.0,US,desktop,0,0.00
3,user_0004,39.0,US,desktop,1,247.43
4,user_0005,295.0,GB,NaN,1,5.96


,session_id,user_id,day_offset,minutes_played,completed
0,session_01326,user_0274,-7,84.25,1
1,session_03494,user_0706,-7,130.05,1
2,session_04282,user_0872,-1,33.65,1
3,session_04588,user_0937,-3,74.44,1
4,session_00441,user_0092,-1,66.84,1


duplicated 0
duplicated 30
out of window 30
label distribution: returned_next_week
0    0.707
1    0.293
Name: proportion, dtype: float64
invalidate duration: 49


In [ ]:
assert players["user_id"].is_unique
assert players["returned_next_week"].notna().all()
assert set(players["returned_next_week"]) == {0, 1}
assert sessions["user_id"].isin(players["user_id"]).all()
print("Input contract checked.")


## Problem 1 · Step 2 — Clean the session log (4–9 min)

Create **clean_sessions** without modifying **sessions**. Remove exact duplicate rows, keep only day offsets -7 through -1, and convert negative durations to missing values. Keep a session with an unknown duration: it still counts as activity. Fail visibly if duplicate session IDs remain with conflicting rows.

**Say before coding:** "I will apply deterministic validity rules now. I will postpone learned imputation statistics until after the train/test split."

**Possible interviewer questions**

1. Why not use dropna() on the whole table?
2. Why drop exact duplicates rather than silently keep the first session ID?
3. Is this cleaning before splitting a leak?


In [ ]:
# Step 2: write your implementation here.
# Create the variables named in the task above.
clean_sessions = sessions.drop_duplicates().copy()

masked = sessions['day_offset'].between(-7,-1)

clean_sessions = clean_sessions.loc[masked].copy()

masked = clean_sessions['minutes_played'].lt(0)
clean_sessions.loc[masked,'minutes_played'] = np.nan



#raise NotImplementedError("Complete Step 2, then run the checks below.")


In [ ]:
assert clean_sessions["session_id"].is_unique
assert clean_sessions["day_offset"].between(-7, -1).all()
assert clean_sessions["minutes_played"].dropna().ge(0).all()
assert clean_sessions["minutes_played"].isna().any()
print("Historical-session checks passed.")


## Problem 1 · Step 3 — Aggregate and join (9–14 min)

Build **player_features**, one row for every player. Add **session_count**, **total_minutes**, **avg_minutes**, and **completion_rate** from clean history. Keep a duration-observation count to distinguish unknown durations from inactivity. For a player with no sessions, use 0 for these activity features. For a player with sessions but no valid durations, keep total and average minutes missing.

Report (1) the fraction of players with no historical activity, and (2) average per-player observed total minutes by device. Include inactive players; keep an explicit missing-device group. Counts accompanying the averages should show missing-duration coverage.

**Say before coding:** "I will aggregate before joining, so players with many sessions do not accidentally receive more training weight. A left join preserves inactive players."

**Possible interviewer questions**

1. Why size rather than count for the number of sessions?
2. What happens if you join players directly to all sessions?
3. Does higher play time on one device prove that device increases engagement?


In [18]:
# Step 3: write your implementation here.
# Create the variables named in the task above.
activity = clean_sessions.groupby('user_id').agg(
    session_count = ("session_id", "size"),
    total_minutes = ("minutes_played", "sum"),
    avg_minutes = ("minutes_played", "mean"),
    completation_rate = ("completed", 'mean'),
    known_duration_count=("minutes_played", "count"),
)
activity.loc[activity["known_duration_count"].eq(0), "total_minutes"] = np.nan
player_features = players.merge(activity, on='user_id', how='left', validate='one_to_one' )

no_sessions = player_features['session_count'].isna()
player_features.loc[no_sessions, activity.columns] = 0





#raise NotImplementedError("Complete Step 3, then run the checks below.")


In [ ]:
assert len(player_features) == len(players)
assert player_features["user_id"].is_unique
assert player_features["session_count"].notna().all()
assert player_features["session_count"].sum() == len(clean_sessions)
inactive = player_features["session_count"].eq(0)
assert inactive.any()
assert player_features.loc[inactive, ["total_minutes", "avg_minutes"]].eq(0).all().all()
unknown = player_features["session_count"].gt(0) & player_features["known_duration_count"].eq(0)
assert unknown.any()
assert player_features.loc[unknown, ["total_minutes", "avg_minutes"]].isna().all().all()
print("One-row-per-player and missing-versus-zero checks passed.")


## Problem 2 · Step 4 — Select features and split (14–18 min)

Define **numeric_features**, **categorical_features**, and **feature_cols**. Use the five numeric predictors account_age_days, session_count, total_minutes, avg_minutes, completion_rate, plus country and device. Create **X** and **y**, then an 80/20 stratified split: **X_train**, **X_test**, **y_train**, **y_test**. Use SEED. Keep user_id as the row index for auditing, not a predictor.

**Say before coding:** "My feature list is explicit. I will reserve the test set now and fit every learned preprocessing step only on training data."

**Possible interviewer questions**

1. Why use stratify=y?
2. Why is a random split acceptable here, and when is it wrong?
3. Is total_minutes already leaking because we computed it before the split?


In [20]:
# Step 4: write your implementation here.
# Create the variables named in the task above.
numeric_features = [
    "account_age_days", "session_count", "total_minutes", "avg_minutes", "completation_rate"
]
categorical_features = ["country", "device"]
feature_cols = numeric_features + categorical_features

X = player_features[feature_cols]
y = player_features['returned_next_week']

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=SEED)


#raise NotImplementedError("Complete Step 4, then run the checks below.")


In [ ]:
assert not {"user_id", "returned_next_week", "next_week_minutes"}.intersection(feature_cols)
assert X_train.index.intersection(X_test.index).empty
assert len(X_train) + len(X_test) == len(player_features)
assert X_train.index.equals(y_train.index) and X_test.index.equals(y_test.index)
assert set(y_train) == set(y_test) == {0, 1}
print("Split and feature-boundary checks passed.")


## Problem 2 · Step 5 — Preprocess and train (18–27 min)

Fit **baseline**, a DummyClassifier using the training-label prior. Then fit **model**, one Pipeline containing a ColumnTransformer and LogisticRegression. Numeric columns: median imputation, then standardization. Categorical columns: most-frequent imputation, then one-hot encoding that tolerates unseen categories. Fit only X_train and y_train. Keep default regularization; set max_iter=500.

**Say before coding:** "A dummy baseline tells me whether I learned more than the class frequency. Logistic regression is fast, interpretable, and a reasonable first model for mixed tabular features."

**Possible interviewer questions**

1. What does Pipeline prevent?
2. Why scale numeric features for logistic regression?
3. What is logistic regression optimizing?


In [22]:
# Step 5: write your implementation here.
# Create the variables named in the task above.
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])
preprocess = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

baseline = DummyClassifier(strategy="prior").fit(X_train, y_train)
model = Pipeline([
    ("preprocess", preprocess),
    ("classifier", LogisticRegression(max_iter=500)),
])
model.fit(X_train, y_train)



#raise NotImplementedError("Complete Step 5, then run the checks below.")


Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['account_age_days',
                                                   'session_count',
                                                   'total_minutes',
                                                   'avg_minutes',
                                                   'completation_rate']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['country', 'device'])])),
                ('classifier', LogisticRegression(max_iter=500))])

In [ ]:
assert list(model.classes_) == [0, 1]
fitted_imputer = model.named_steps["preprocess"].named_transformers_["num"].named_steps["imputer"]
np.testing.assert_allclose(fitted_imputer.statistics_, X_train[numeric_features].median().to_numpy())
print("The numeric imputer learned the training medians.")


## Problem 2 · Step 6 — Evaluate (27–32 min)

Create **baseline_prob** and **model_prob** for the positive class on X_test. Build a **metrics** table with accuracy, ROC-AUC, average precision (AP), precision, recall and F1. Use a fixed 0.5 threshold for label metrics; pass probabilities to ROC-AUC and AP. Show a confusion matrix with clearly labeled rows and columns. Briefly compare training and test ROC-AUC. Do not tune against the test results.

**Say before coding:** "I will evaluate ranking with probability scores and decision quality at a stated threshold. Accuracy alone may hide poor recall for returning players."

**Possible interviewer questions**

1. Why not compute ROC-AUC using model.predict()? 
2. What do ROC-AUC and AP tell you?
3. What if the business needs 80% recall instead of a 0.5 threshold?


In [ ]:
# Step 6: write your implementation here.
# Create the variables named in the task above.
baseline_prob = baseline.predict_proba(X_test)[:, 1]
model_prob = model.predict_proba(X_test)[:, 1]
display(baseline.predict_proba(X_test))
print(model.classes_)

def evaluate(y_true, probabilities):
    predicted = (probabilities >= 0.5).astype(int)
    return {
        "accuracy": accuracy_score(y_true, predicted),
        "roc_auc": roc_auc_score(y_true, probabilities),
        "average_precision": average_precision_score(y_true, probabilities),
        "precision@0.5": precision_score(y_true, predicted, zero_division=0),
        "recall@0.5": recall_score(y_true, predicted, zero_division=0),
        "f1@0.5": f1_score(y_true, predicted, zero_division=0),
    }


results = []

results.append(evaluate(y_test, baseline_prob))
results.append(evaluate(y_test, model_prob))

print(pd.DataFrame(results))

#raise NotImplementedError("Complete Step 6, then run the checks below.")


array([[0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.705, 0.295],
       [0.7

[0 1]
   accuracy   roc_auc  average_precision  precision@0.5  recall@0.5   f1@0.5
0     0.715  0.500000           0.285000       0.000000    0.000000  0.00000
1     0.770  0.790578           0.616592       0.622222    0.491228  0.54902


In [ ]:
assert model_prob.shape == (len(X_test),)
assert np.isfinite(model_prob).all() and ((0 <= model_prob) & (model_prob <= 1)).all()
assert np.isclose(metrics.loc["prior_baseline", "roc_auc"], 0.5)
assert np.isclose(metrics.loc["prior_baseline", "average_precision"], y_test.mean())
print("Probability and baseline-metric checks passed. Model scores are observations, not a pass/fail target.")


## Problem 2 · Step 7 — Predict and summarize (32–35 min)

The next cell supplies **new_players** with already-aggregated pre-cutoff features. Compute **new_prob** using the fitted pipeline, without refitting. Include the unknown country and missing attributes. Return the positive-class probability, and give a 30-second summary of your approach, one limitation, and one next experiment.

**Say before coding:** "I will reuse the fitted preprocessing and classifier. The second input deliberately tests missing data and an unseen categorical value."

**Possible interviewer questions**

1. What happens to the unseen country?
2. What would you try next?
3. What must training and serving share?


In [ ]:
new_players = pd.DataFrame([
    {"account_age_days": 15, "session_count": 3, "total_minutes": 45,
     "avg_minutes": 15, "completion_rate": 2/3, "country": "US", "device": "mobile"},
    {"account_age_days": np.nan, "session_count": 0, "total_minutes": 0,
     "avg_minutes": 0, "completion_rate": 0, "country": "NEW_COUNTRY", "device": np.nan},
], index=["new_player_A", "new_player_B"])

display(new_players)


In [ ]:
# Step 7: write your implementation here.
# Create the variables named in the task above.
raise NotImplementedError("Complete Step 7, then run the checks below.")


In [ ]:
assert new_prob.shape == (2,)
assert np.isfinite(new_prob).all() and ((0 <= new_prob) & (new_prob <= 1)).all()
print("Inference handles missing values and an unseen country.")


## If the interviewer extends the problem

1. **Predict next-week minutes instead.** This becomes regression: use next_week_minutes as the target, never as an input. Try a mean baseline and a regularized regressor; compare MAE and RMSE. Discuss the many zeros and long tail. Keep the same feature-time boundary.
2. **Many daily rows per player.** Clarify whether the goal is future predictions for existing players or generalization to unseen players. Use time, group, or a combined split accordingly; mature label windows must not reach into the evaluation period.
3. **A 100 GB session log.** Push filtering/deduplication/aggregation into a database or distributed job. Chunk sums and counts merge easily; an exact distinct-session count needs a deliberate cross-chunk deduplication strategy. Do not average chunk averages without weighting.
4. **A nearly perfect model.** First audit future columns, duplicated entities across subsets, target-derived transformations, and whether preprocessing was fitted on all data. High accuracy is not proof of a good model.

## References and provenance

The interview **format** comes from the description pasted in your [shared conversation](https://chatgpt.com/share/6ad92cab-a9dc-83e8-b72e-654063228d54). The problem, data, code and questions here are original practice material. They are not evidence that Roblox asks this exact task, and this local notebook is not connected to CodeSignal.

Implementation references: [scikit-learn: avoiding data leakage](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage), [ColumnTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), [ROC-AUC](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html), [pandas merge validation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html).
